# De coocurrencias a embeddings: cómo el contexto empieza a volverse semántica

**Tecnicatura Superior en Ciencias de Datos e Inteligencia Artificial**  
**Procesamiento de Lenguaje Natural**

---

## Objetivo

En los cuadernos anteriores trabajaste con `Bag of Words`, `TF-IDF` y vectores semánticos preentrenados. Ya viste que las representaciones sparse permiten contar y ponderar términos, y también viste que los vectores densos pueden capturar relaciones semánticas que esos conteos no modelan bien.

En este cuaderno vamos a construir el puente entre esos dos mundos. Todavía no vas a entrenar `Word2Vec`, `FastText` ni `GloVe`. Primero vamos a responder una pregunta más básica y más importante: **¿qué cambia cuando dejamos de mirar palabras aisladas y empezamos a mirar contextos?**

## Resultados de aprendizaje

Al finalizar este cuaderno vas a poder:

- explicar qué es una matriz de coocurrencias;
- describir la hipótesis distribucional con ejemplos concretos;
- construir perfiles de contexto para palabras;
- usar `PPMI` para ponderar asociaciones distribucionales;
- reducir una representación sparse a vectores densos con `TruncatedSVD`;
- argumentar qué conecta este recorrido con `Word2Vec`, `FastText` y `GloVe`.

## Relación con cuadernos anteriores

- En `005_Vectorizacion` trabajaste con `Bag of Words`, `TF-IDF` y n-gramas.
- En `006_lab_integrador_guiado` usaste esas representaciones sobre un corpus pequeño y comparable.
- En `008/De_Text_Mining_a_Representaciones_Semántica.ipynb` viste por qué los vectores densos cambian la discusión semántica.
- **Acá** vas a ver cómo el contexto empieza a producir esa geometría.


---

## 1. Preparación del entorno

Vamos a trabajar con un conjunto pequeño de herramientas. La idea no es ocultar la mecánica detrás de una librería especializada, sino construirla paso a paso con código legible.


In [ ]:
# --- Importaciones del cuaderno ---

# re: limpieza y tokenización simple basada en expresiones regulares.
import re

# math: logaritmos para el cálculo de PMI y PPMI.
import math

# defaultdict: diccionarios con valores iniciales automáticos para los conteos.
from collections import defaultdict

# numpy: operaciones numéricas sobre matrices.
import numpy as np

# pandas: tablas legibles para inspeccionar matrices y resultados.
import pandas as pd

# HTML y display: salidas más legibles dentro del notebook.
from IPython.display import HTML, display

# CountVectorizer: recordatorio breve del problema de BoW.
from sklearn.feature_extraction.text import CountVectorizer

# cosine_similarity: comparación geométrica entre filas y vectores.
from sklearn.metrics.pairwise import cosine_similarity

# TruncatedSVD: reducción de dimensionalidad para pasar de sparse a denso.
from sklearn.decomposition import TruncatedSVD

# matplotlib y seaborn: visualizaciones.
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual consistente con el resto del repositorio.
sns.set_theme(style="ticks", context="notebook", palette="colorblind", font_scale=1.0)
PALETA = sns.color_palette("colorblind")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["axes.titlepad"] = 14
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 3)

print("Entorno listo para trabajar con coocurrencias y vectores densos.")


In [ ]:
# --- Señalética visual breve para orientar la lectura ---

tarjeta_lectura = """
<div style='max-width: 920px; margin: 0.4rem 0 1.2rem 0; padding: 0.9rem 1rem; border: 1px solid #c7d8e8; border-radius: 10px; background: linear-gradient(180deg, #f7fbff 0%, #eef6fb 100%);'>
  <div style='font-size: 1.02rem; font-weight: 700; margin-bottom: 0.45rem;'>Cómo conviene leer este cuaderno</div>
  <div style='line-height: 1.65;'>
    <strong>1.</strong> Primero mirá el ejemplo manual.<br>
    <strong>2.</strong> Después seguí cómo esa lógica se vuelve una matriz.<br>
    <strong>3.</strong> Recién al final mirá el paso a vectores densos y la conexión con <code>Word2Vec</code>, <code>GloVe</code> y <code>FastText</code>.
  </div>
</div>
"""

display(HTML(tarjeta_lectura))


## Cómo conviene leer este cuaderno

Este material está pensado para una lectura en tres tiempos:

- primero una intuición corta;
- después un ejemplo manual completo;
- por último la automatización y el pasaje a geometría.

Si una tabla se vuelve densa, no hace falta leerla de punta a punta. Conviene mirar **qué palabras aparecen cerca**, **qué asociaciones suben de peso** y **qué cambia cuando pasamos a vectores densos**.


---

## 2. Concepto: del término aislado al perfil de contexto

`Bag of Words` y `TF-IDF` describen documentos a partir de términos observados. Eso permite contar, ponderar y comparar textos. Pero sigue habiendo una limitación fuerte: cada palabra queda tratada como una unidad separada.

Si dos palabras no comparten forma exacta, esas representaciones no tienen una manera directa de registrar que pueden ocupar lugares parecidos en el lenguaje. Por eso ahora vamos a cambiar la pregunta.

En vez de preguntar:

- ¿cuántas veces aparece esta palabra?

vamos a empezar a preguntar:

- ¿con qué palabras suele aparecer esta palabra?

Ese cambio se apoya en la **hipótesis distribucional**:

> Una palabra se caracteriza, en parte, por los contextos en los que aparece.

Si representamos a una palabra por las palabras que la rodean, obtenemos un perfil de contexto:

$$
\mathbf{v}(w) = \big(c(w, c_1), c(w, c_2), \dots, c(w, c_m)\big)
$$

donde:

- `$w$` es la palabra objetivo;
- `$c_i$` es una palabra de contexto posible;
- `$c(w, c_i)$` es la cantidad de veces que `$c_i$` aparece cerca de `$w$`.


Antes de automatizar nada, conviene reabrir el problema con un ejemplo muy breve.


In [ ]:
# --- Recordatorio breve: BoW distingue formas, no entornos ---

frases_recordatorio = [
    "Mi papá me abrazó antes del partido",
    "Mi padre me abrazó antes del encuentro",
    "La hinchada gritó el gol en la cancha",
]

vectorizador_bow = CountVectorizer()
matriz_bow_recordatorio = vectorizador_bow.fit_transform(frases_recordatorio)
vocabulario_bow_recordatorio = vectorizador_bow.get_feature_names_out()

tabla_bow_recordatorio = pd.DataFrame(
    matriz_bow_recordatorio.toarray(),
    columns=vocabulario_bow_recordatorio,
    index=["Frase 1", "Frase 2", "Frase 3"]
)

print("Representación BoW de tres frases breves")
display(tabla_bow_recordatorio)


### Qué conviene mirar en este bloque

Antes de seguir, fijate en dos cosas:

- `papá` y `padre` aparecen en columnas distintas;
- `partido` y `encuentro` también quedan separados, aunque el sentido de la frase sea parecido.

Esa separación es exactamente el límite que vamos a empezar a corregir.


La tabla vuelve visible un límite conocido: `papá` y `padre` quedan en columnas diferentes, del mismo modo que `partido` y `encuentro`. Si queremos modelar cercanía semántica, necesitamos dejar de mirar solo palabras exactas y empezar a mirar **entornos de uso**.


---

## 3. Ejercicio manual: una ventana de contexto antes de automatizar

Antes de construir una matriz completa, vamos a hacer el mecanismo a mano. Usamos una ventana local: para cada palabra objetivo miramos una cantidad fija de tokens a izquierda y derecha.

Si el tamaño de la ventana es `$k = 1$`, el contexto de una palabra en la posición `$t$` queda así:

$$
\text{contexto}(w_t) = \{w_{t-1}, w_{t+1}\}
$$

En este paso vamos a inspeccionar manualmente qué rodea a la palabra `hija`.


In [ ]:
# --- Oraciones manuales para inspección humana completa ---

oraciones_manuales_tokenizadas = [
    ["papá", "abrazó", "hija", "casa"],
    ["padre", "acompañó", "hija", "hogar"],
    ["madre", "cuidó", "hija", "casa"],
]

cantidad_oraciones_manuales = len(oraciones_manuales_tokenizadas)
print("Oraciones manuales tokenizadas")
print("-" * 40)

for indice_oracion in range(cantidad_oraciones_manuales):
    numero_oracion = indice_oracion + 1
    tokens_oracion = oraciones_manuales_tokenizadas[indice_oracion]
    print(f"Oración {numero_oracion}: {tokens_oracion}")


Ahora vamos a recorrer esas oraciones y a registrar, una por una, las palabras que aparecen al lado de `hija`. La idea es que el conteo quede completamente visible.


In [ ]:
# --- Conteo manual de contextos para la palabra 'hija' ---

palabra_objetivo_manual = "hija"
ventana_manual = 1
registros_contexto_manual = []
conteos_contexto_manual = defaultdict(int)

for indice_oracion in range(cantidad_oraciones_manuales):
    tokens_oracion = oraciones_manuales_tokenizadas[indice_oracion]
    cantidad_tokens_oracion = len(tokens_oracion)

    for posicion_token in range(cantidad_tokens_oracion):
        token_actual = tokens_oracion[posicion_token]

        if token_actual == palabra_objetivo_manual:
            inicio_ventana = posicion_token - ventana_manual
            fin_ventana = posicion_token + ventana_manual

            for posicion_contexto in range(inicio_ventana, fin_ventana + 1):
                if posicion_contexto < 0:
                    continue

                if posicion_contexto >= cantidad_tokens_oracion:
                    continue

                if posicion_contexto == posicion_token:
                    continue

                palabra_contexto = tokens_oracion[posicion_contexto]
                conteos_contexto_manual[palabra_contexto] += 1

                registro_actual = {
                    "oración": indice_oracion + 1,
                    "palabra_objetivo": palabra_objetivo_manual,
                    "palabra_contexto": palabra_contexto,
                }
                registros_contexto_manual.append(registro_actual)

tabla_registros_contexto_manual = pd.DataFrame(registros_contexto_manual)
display(tabla_registros_contexto_manual)

filas_conteo_manual = []
for palabra_contexto in conteos_contexto_manual:
    fila_actual = {
        "palabra_contexto": palabra_contexto,
        "frecuencia": conteos_contexto_manual[palabra_contexto],
    }
    filas_conteo_manual.append(fila_actual)

tabla_conteo_manual = pd.DataFrame(filas_conteo_manual)
tabla_conteo_manual = tabla_conteo_manual.sort_values(by="frecuencia", ascending=False)

print("\nConteo manual de contextos para 'hija'")
display(tabla_conteo_manual)


> **Pausa breve de lectura**
>
> En esta etapa no estás buscando semántica compleja. Solo estás registrando un cambio de representación: la palabra `hija` dejó de ser una forma aislada y pasó a estar descrita por las palabras que la rodean.


En este ejemplo aparece una idea decisiva: `hija` no quedó representada por su ortografía ni por su frecuencia global, sino por las palabras que la rodearon. Ese es el germen de toda representación distribucional.


---

## 4. Automatización: del contexto local a la matriz global

Ahora vamos a pasar del ejemplo manual a un corpus completo, pero todavía controlado. Elegimos tres campos semánticos simples para que la inspección siga siendo posible:

- familia;
- fútbol;
- escritura.

Este corpus no busca producir un modelo potente. Busca volver visible el mecanismo.


In [ ]:
# --- Corpus controlado del cuaderno ---

corpus_controlado = [
    "Mi papá abrazó a mi hija en casa.",
    "Mi padre abrazó a su hija en el hogar.",
    "La familia volvió a casa con la nena dormida.",
    "El hogar quedó en silencio cuando la hija se durmió.",
    "La hinchada gritó el gol en la cancha.",
    "La tribuna festejó el partido en la cancha.",
    "El jugador pateó la pelota y llegó el gol.",
    "El partido dejó cantando a toda la hinchada.",
    "El escritor corrigió el relato en la revista.",
    "La autora publicó la historia en la revista.",
    "El texto del relato llegó a los lectores.",
    "La historia conmovió a los lectores del barrio.",
]

cantidad_textos_controlados = len(corpus_controlado)
print(f"Cantidad de textos del corpus controlado: {cantidad_textos_controlados}")
print("")

for indice_texto in range(cantidad_textos_controlados):
    numero_texto = indice_texto + 1
    texto_actual = corpus_controlado[indice_texto]
    print(f"Texto {numero_texto}: {texto_actual}")


### 4.1 Tokenización simple y consistente

En este cuaderno no necesitamos un análisis lingüístico complejo. Nos alcanza con una tokenización simple y estable que nos permita construir ventanas de contexto.

Vamos a pasar todo a minúscula, a extraer secuencias alfabéticas y a remover algunas stopwords mínimas para que la matriz no quede dominada por palabras funcionales.


In [ ]:
# --- Stopwords mínimas para mantener la matriz legible ---

STOPWORDS_MINIMAS = [
    "a",
    "al",
    "con",
    "de",
    "del",
    "el",
    "en",
    "la",
    "los",
    "mi",
    "su",
    "toda",
    "y",
]

def tokenizar_simple(texto, stopwords_minimas):
    """Limpia un texto, extrae tokens alfabéticos y remueve stopwords mínimas."""

    # Pasamos todo a minúscula para unificar variantes ortográficas básicas.
    texto_en_minusculas = texto.lower()

    # Extraemos solo secuencias de letras, incluyendo vocales acentuadas y ñ.
    tokens_crudos = re.findall(r"[a-záéíóúñü]+", texto_en_minusculas)

    # Construimos la lista final paso a paso.
    tokens_limpios = []

    for token_actual in tokens_crudos:
        if token_actual in stopwords_minimas:
            continue

        tokens_limpios.append(token_actual)

    return tokens_limpios

def tokenizar_corpus(textos_corpus, stopwords_minimas):
    """Aplica la tokenización simple a cada texto del corpus y devuelve una lista de listas."""

    corpus_tokenizado = []

    for indice_texto in range(len(textos_corpus)):
        texto_actual = textos_corpus[indice_texto]
        tokens_texto_actual = tokenizar_simple(texto_actual, stopwords_minimas)
        corpus_tokenizado.append(tokens_texto_actual)

    return corpus_tokenizado

corpus_controlado_tokenizado = tokenizar_corpus(corpus_controlado, STOPWORDS_MINIMAS)

filas_corpus_tokenizado = []
for indice_texto in range(len(corpus_controlado_tokenizado)):
    fila_actual = {
        "texto": indice_texto + 1,
        "tokens": corpus_controlado_tokenizado[indice_texto],
    }
    filas_corpus_tokenizado.append(fila_actual)

tabla_corpus_tokenizado = pd.DataFrame(filas_corpus_tokenizado)
display(tabla_corpus_tokenizado)


### 4.2 La ventana de contexto como unidad de observación

Ahora formalizamos la idea de contexto local. Si la ventana es de tamaño `$k = 2$`, para una palabra en la posición `$t$` observamos hasta dos tokens a izquierda y dos a derecha:

$$
\text{contexto}(w_t) = \{w_{t-2}, w_{t-1}, w_{t+1}, w_{t+2}\}
$$

Vamos a mirar primero un ejemplo aislado antes de construir toda la matriz.


In [ ]:
# --- Función para inspeccionar contextos de una palabra objetivo ---

def extraer_contextos_de_palabra(tokens_texto, palabra_objetivo, tamaño_ventana=2):
    """Devuelve una tabla con los contextos locales observados para una palabra dentro de un texto tokenizado."""

    registros_contexto = []
    cantidad_tokens = len(tokens_texto)

    for posicion_token in range(cantidad_tokens):
        palabra_actual = tokens_texto[posicion_token]

        if palabra_actual != palabra_objetivo:
            continue

        inicio_ventana = posicion_token - tamaño_ventana
        fin_ventana = posicion_token + tamaño_ventana

        for posicion_contexto in range(inicio_ventana, fin_ventana + 1):
            if posicion_contexto < 0:
                continue

            if posicion_contexto >= cantidad_tokens:
                continue

            if posicion_contexto == posicion_token:
                continue

            palabra_contexto = tokens_texto[posicion_contexto]

            registro_actual = {
                "palabra_objetivo": palabra_objetivo,
                "palabra_contexto": palabra_contexto,
                "distancia": posicion_contexto - posicion_token,
            }
            registros_contexto.append(registro_actual)

    tabla_contextos = pd.DataFrame(registros_contexto)
    return tabla_contextos

texto_ejemplo_contexto = corpus_controlado_tokenizado[4]
tabla_contextos_gol = extraer_contextos_de_palabra(texto_ejemplo_contexto, "gol", tamaño_ventana=2)

print("Texto tokenizado de ejemplo")
print(texto_ejemplo_contexto)
print("")
print("Contextos observados para 'gol' con ventana = 2")
display(tabla_contextos_gol)


Cada una de esas observaciones puede leerse de dos maneras:

- como una entrada en una matriz de coocurrencias;
- como un ejemplo de entrenamiento para un modelo predictivo.

Por ahora vamos por la primera ruta: la ruta basada en conteos.


In [ ]:
# --- Construcción explícita de la matriz de coocurrencias ---

def construir_tabla_coocurrencias(corpus_tokenizado, tamaño_ventana=2):
    """Construye una tabla de coocurrencias palabra-contexto a partir de un corpus tokenizado."""

    # Primero reunimos el vocabulario completo del corpus.
    vocabulario_unico = set()

    for indice_texto in range(len(corpus_tokenizado)):
        tokens_texto = corpus_tokenizado[indice_texto]

        for token_actual in tokens_texto:
            vocabulario_unico.add(token_actual)

    palabras_del_vocabulario = sorted(vocabulario_unico)

    # Creamos un índice para ubicar rápidamente cada palabra en la matriz.
    indice_por_palabra = {}
    for indice_palabra in range(len(palabras_del_vocabulario)):
        palabra_actual = palabras_del_vocabulario[indice_palabra]
        indice_por_palabra[palabra_actual] = indice_palabra

    cantidad_palabras = len(palabras_del_vocabulario)
    matriz_coocurrencias = np.zeros((cantidad_palabras, cantidad_palabras), dtype=int)

    # Recorremos cada texto y cada posición para sumar coocurrencias dentro de la ventana.
    for indice_texto in range(len(corpus_tokenizado)):
        tokens_texto = corpus_tokenizado[indice_texto]
        cantidad_tokens = len(tokens_texto)

        for posicion_token in range(cantidad_tokens):
            palabra_objetivo = tokens_texto[posicion_token]
            fila_objetivo = indice_por_palabra[palabra_objetivo]

            inicio_ventana = posicion_token - tamaño_ventana
            fin_ventana = posicion_token + tamaño_ventana

            for posicion_contexto in range(inicio_ventana, fin_ventana + 1):
                if posicion_contexto < 0:
                    continue

                if posicion_contexto >= cantidad_tokens:
                    continue

                if posicion_contexto == posicion_token:
                    continue

                palabra_contexto = tokens_texto[posicion_contexto]
                columna_contexto = indice_por_palabra[palabra_contexto]
                matriz_coocurrencias[fila_objetivo, columna_contexto] += 1

    tabla_coocurrencias = pd.DataFrame(
        matriz_coocurrencias,
        index=palabras_del_vocabulario,
        columns=palabras_del_vocabulario,
    )

    return tabla_coocurrencias

tabla_coocurrencias = construir_tabla_coocurrencias(corpus_controlado_tokenizado, tamaño_ventana=2)
cantidad_filas_coocurrencias = tabla_coocurrencias.shape[0]
cantidad_columnas_coocurrencias = tabla_coocurrencias.shape[1]

print(f"Tamaño de la tabla de coocurrencias: {cantidad_filas_coocurrencias} x {cantidad_columnas_coocurrencias}")


Vamos a mirar una porción de la matriz para que la idea siga siendo legible.


In [ ]:
# --- Submatriz legible para inspección humana ---

palabras_seleccionadas = [
    "papá",
    "padre",
    "hija",
    "casa",
    "hogar",
    "gol",
    "partido",
    "hinchada",
    "cancha",
    "tribuna",
    "relato",
    "historia",
    "revista",
    "lectores",
]

filas_presentes = []
columnas_presentes = []
indice_completo = list(tabla_coocurrencias.index)
columnas_completas = list(tabla_coocurrencias.columns)

for palabra_actual in palabras_seleccionadas:
    if palabra_actual in indice_completo:
        filas_presentes.append(palabra_actual)

    if palabra_actual in columnas_completas:
        columnas_presentes.append(palabra_actual)

submatriz_coocurrencias = tabla_coocurrencias.loc[filas_presentes, columnas_presentes]
display(submatriz_coocurrencias)


### Cómo leer esta submatriz

No hace falta recorrer cada celda. Conviene mirar comparaciones puntuales:

- `papá` y `padre`;
- `casa` y `hogar`;
- `relato` y `historia`;
- `gol`, `partido`, `hinchada` y `cancha`.

Si varias filas comparten contextos parecidos, ya está apareciendo la lógica distribucional.


Cada fila de esa tabla es un **perfil de contexto**. Dos palabras no tienen que ser iguales ni aparecer juntas siempre para parecerse: basta con que sus filas se parezcan.


In [ ]:
# --- Similitud coseno entre perfiles de contexto ---

def calcular_similitud_coseno_entre_palabras(tabla_perfiles, palabra_1, palabra_2):
    """Calcula la similitud coseno entre dos palabras a partir de sus filas en una tabla de perfiles."""

    fila_palabra_1 = tabla_perfiles.loc[[palabra_1]]
    fila_palabra_2 = tabla_perfiles.loc[[palabra_2]]

    matriz_similitud = cosine_similarity(fila_palabra_1, fila_palabra_2)
    similitud_final = float(matriz_similitud[0, 0])

    return similitud_final

pares_para_comparar = [
    ("papá", "padre"),
    ("casa", "hogar"),
    ("gol", "partido"),
    ("relato", "historia"),
    ("papá", "cancha"),
    ("historia", "tribuna"),
]

registros_similitud = []
for indice_par in range(len(pares_para_comparar)):
    palabra_1 = pares_para_comparar[indice_par][0]
    palabra_2 = pares_para_comparar[indice_par][1]
    similitud_actual = calcular_similitud_coseno_entre_palabras(tabla_coocurrencias, palabra_1, palabra_2)

    registro_actual = {
        "palabra_1": palabra_1,
        "palabra_2": palabra_2,
        "similitud_coseno": round(similitud_actual, 4),
    }
    registros_similitud.append(registro_actual)

tabla_similitud_coocurrencias = pd.DataFrame(registros_similitud)
display(tabla_similitud_coocurrencias)


> **Pausa de observación**
>
> Si la similitud sube entre `papá` y `padre`, o entre `casa` y `hogar`, no es porque las palabras sean iguales. Sube porque sus perfiles de contexto empiezan a parecerse.


Ya apareció un cambio fuerte respecto de BoW: la comparación no depende solo de compartir palabras exactas, sino de tener **entornos parecidos**. Pero todavía no toda coocurrencia vale lo mismo.


---

## 5. Ponderación distribucional: no toda coocurrencia vale lo mismo

La matriz de coocurrencias mejora mucho respecto de BoW, pero conserva un problema: algunas palabras aparecen en muchos contextos y tienden a dominar el conteo.

Necesitamos una ponderación que destaque asociaciones más informativas y reduzca el peso de coocurrencias triviales. Para eso vamos a usar `PMI` y su variante positiva, `PPMI`.

$$
PMI(w, c) = \log_2 \frac{P(w, c)}{P(w) \cdot P(c)}
$$

Si la asociación entre `$w$` y `$c$` es más fuerte de lo esperable por azar, el valor sube. Si no aporta información, el valor baja.

Como los valores negativos suelen ser poco útiles en este contexto, usamos:

$$
PPMI(w, c) = \max(PMI(w, c), 0)
$$


In [ ]:
# --- Cálculo explícito de la matriz PPMI ---

def calcular_tabla_ppmi(tabla_coocurrencias):
    """Convierte una tabla de coocurrencias en una tabla ponderada con PPMI."""

    matriz_coocurrencias = tabla_coocurrencias.to_numpy(dtype=float)
    suma_total_coocurrencias = matriz_coocurrencias.sum()

    sumas_filas = matriz_coocurrencias.sum(axis=1)
    sumas_columnas = matriz_coocurrencias.sum(axis=0)

    matriz_ppmi = np.zeros_like(matriz_coocurrencias, dtype=float)

    cantidad_filas = matriz_coocurrencias.shape[0]
    cantidad_columnas = matriz_coocurrencias.shape[1]

    for indice_fila in range(cantidad_filas):
        for indice_columna in range(cantidad_columnas):
            frecuencia_conjunta = matriz_coocurrencias[indice_fila, indice_columna]

            if frecuencia_conjunta == 0:
                continue

            probabilidad_conjunta = frecuencia_conjunta / suma_total_coocurrencias
            probabilidad_fila = sumas_filas[indice_fila] / suma_total_coocurrencias
            probabilidad_columna = sumas_columnas[indice_columna] / suma_total_coocurrencias

            denominador_independencia = probabilidad_fila * probabilidad_columna

            if denominador_independencia == 0:
                continue

            valor_pmi = math.log2(probabilidad_conjunta / denominador_independencia)

            if valor_pmi < 0:
                valor_pmi = 0

            matriz_ppmi[indice_fila, indice_columna] = valor_pmi

    tabla_ppmi = pd.DataFrame(
        matriz_ppmi,
        index=tabla_coocurrencias.index,
        columns=tabla_coocurrencias.columns,
    )

    return tabla_ppmi

tabla_ppmi = calcular_tabla_ppmi(tabla_coocurrencias)
print("Matriz PPMI calculada correctamente.")


Para que la diferencia se vea mejor, conviene comparar los contextos más importantes antes y después de ponderar.


In [ ]:
# --- Comparación entre conteos crudos y PPMI ---

def extraer_top_contextos(tabla_perfiles, palabra_objetivo, top_n=5, nombre_valor="valor"):
    """Extrae los contextos no nulos con mayor valor para una palabra objetivo."""

    fila_palabra = tabla_perfiles.loc[palabra_objetivo]
    registros_contexto = []

    for palabra_contexto in fila_palabra.index:
        valor_actual = fila_palabra[palabra_contexto]

        if valor_actual <= 0:
            continue

        registro_actual = {
            "palabra_contexto": palabra_contexto,
            nombre_valor: float(valor_actual),
        }
        registros_contexto.append(registro_actual)

    tabla_contextos = pd.DataFrame(registros_contexto)
    tabla_contextos = tabla_contextos.sort_values(by=nombre_valor, ascending=False)
    tabla_contextos = tabla_contextos.head(top_n)

    return tabla_contextos

palabras_para_comparar_contextos = ["gol", "papá"]
tablas_comparativas = []

for indice_palabra in range(len(palabras_para_comparar_contextos)):
    palabra_actual = palabras_para_comparar_contextos[indice_palabra]

    tabla_cruda = extraer_top_contextos(
        tabla_coocurrencias,
        palabra_actual,
        top_n=5,
        nombre_valor="conteo",
    )
    tabla_cruda["tipo"] = "coocurrencia"
    tabla_cruda["palabra_objetivo"] = palabra_actual

    tabla_ponderada = extraer_top_contextos(
        tabla_ppmi,
        palabra_actual,
        top_n=5,
        nombre_valor="ppmi",
    )
    tabla_ponderada["tipo"] = "ppmi"
    tabla_ponderada["palabra_objetivo"] = palabra_actual

    tablas_comparativas.append(tabla_cruda)
    tablas_comparativas.append(tabla_ponderada)

tabla_contextos_comparados = pd.concat(tablas_comparativas, ignore_index=True)
display(tabla_contextos_comparados)


### Qué conviene comparar entre coocurrencias y PPMI

La lectura importante de esta tabla no es solo cuáles contextos aparecen, sino cuáles **suben de jerarquía** cuando dejamos de mirar conteos crudos y pasamos a una ponderación distribucional.


La diferencia es importante: `PPMI` no inventa asociaciones, pero sí redistribuye el peso de la matriz para resaltar relaciones más específicas. En espíritu, este paso cumple un rol parecido al de `TF-IDF`: la frecuencia sola no alcanza.


---

## 6. De una matriz sparse a vectores densos

Hasta acá cada palabra ya tiene un perfil de contexto, pero sigue siendo una representación de alta dimensionalidad y bastante dispersa. Ahora vamos a comprimir esa información con `TruncatedSVD`.

La idea no es obtener una semántica perfecta. La idea es conservar parte de la estructura relacional en un espacio más pequeño y más operable.


In [ ]:
# --- Reducción de dimensionalidad sobre la matriz PPMI ---

cantidad_dimensiones_densas = 4
modelo_reduccion = TruncatedSVD(n_components=cantidad_dimensiones_densas, random_state=42)
matriz_vectores_densos = modelo_reduccion.fit_transform(tabla_ppmi)

nombres_columnas_densas = []
for indice_dimension in range(cantidad_dimensiones_densas):
    nombre_dimension = f"dimensión_{indice_dimension + 1}"
    nombres_columnas_densas.append(nombre_dimension)

tabla_vectores_densos = pd.DataFrame(
    matriz_vectores_densos,
    index=tabla_ppmi.index,
    columns=nombres_columnas_densas,
)

print("Primeras filas del espacio denso")
display(tabla_vectores_densos.head(10))


### Qué conviene mirar en el espacio denso

En esta etapa ya no leas columnas aisladas. Mirá vecinos y agrupamientos. La pregunta cambió:

- antes: ¿qué términos aparecen?
- ahora: ¿qué palabras ocupan posiciones parecidas en el espacio?


Ahora las palabras ya no están definidas por columnas aisladas de la matriz original, sino por posiciones en un espacio vectorial comprimido. Eso nos permite buscar vecinos semánticos.


In [ ]:
# --- Búsqueda de vecinos en el espacio denso ---

def buscar_vecinos_semanticos(tabla_vectores, palabra_objetivo, top_n=5):
    """Busca las palabras más cercanas a una palabra objetivo dentro de un espacio vectorial denso."""

    fila_objetivo = tabla_vectores.loc[[palabra_objetivo]]
    similitudes = cosine_similarity(fila_objetivo, tabla_vectores)
    vector_similitudes = similitudes[0]

    registros_vecinos = []
    palabras_del_espacio = list(tabla_vectores.index)

    for indice_palabra in range(len(palabras_del_espacio)):
        palabra_actual = palabras_del_espacio[indice_palabra]

        if palabra_actual == palabra_objetivo:
            continue

        similitud_actual = float(vector_similitudes[indice_palabra])
        registro_actual = {
            "palabra_vecina": palabra_actual,
            "similitud_coseno": similitud_actual,
        }
        registros_vecinos.append(registro_actual)

    tabla_vecinos = pd.DataFrame(registros_vecinos)
    tabla_vecinos = tabla_vecinos.sort_values(by="similitud_coseno", ascending=False)
    tabla_vecinos = tabla_vecinos.head(top_n)

    return tabla_vecinos

palabras_para_vecinos = ["papá", "gol", "relato"]

for indice_palabra in range(len(palabras_para_vecinos)):
    palabra_actual = palabras_para_vecinos[indice_palabra]
    tabla_vecinos_actual = buscar_vecinos_semanticos(tabla_vectores_densos, palabra_actual, top_n=5)

    print(f"Vecinos más cercanos a '{palabra_actual}'")
    display(tabla_vecinos_actual)


In [ ]:
# --- Visualización 2D de una parte del espacio denso ---

palabras_para_grafico = [
    "papá",
    "padre",
    "hija",
    "casa",
    "hogar",
    "gol",
    "partido",
    "hinchada",
    "tribuna",
    "cancha",
    "relato",
    "historia",
    "revista",
    "lectores",
]

grupo_por_palabra = {
    "papá": "familia",
    "padre": "familia",
    "hija": "familia",
    "casa": "familia",
    "hogar": "familia",
    "gol": "fútbol",
    "partido": "fútbol",
    "hinchada": "fútbol",
    "tribuna": "fútbol",
    "cancha": "fútbol",
    "relato": "escritura",
    "historia": "escritura",
    "revista": "escritura",
    "lectores": "escritura",
}

color_por_grupo = {
    "familia": PALETA[0],
    "fútbol": PALETA[1],
    "escritura": PALETA[2],
}

fig, ax = plt.subplots(figsize=(11, 7.5), constrained_layout=True)

for indice_palabra in range(len(palabras_para_grafico)):
    palabra_actual = palabras_para_grafico[indice_palabra]
    coordenadas_palabra = tabla_vectores_densos.loc[palabra_actual]
    coordenada_x = coordenadas_palabra["dimensión_1"]
    coordenada_y = coordenadas_palabra["dimensión_2"]
    grupo_actual = grupo_por_palabra[palabra_actual]
    color_actual = color_por_grupo[grupo_actual]

    ax.scatter(
        coordenada_x,
        coordenada_y,
        s=85,
        color=color_actual,
        edgecolor="white",
        linewidth=0.8,
        alpha=0.95,
    )
    ax.text(coordenada_x + 0.02, coordenada_y + 0.02, palabra_actual, fontsize=10)

ax.grid(alpha=0.22, linewidth=0.6)
ax.set_title("Espacio denso derivado de PPMI + SVD", loc="left", fontweight="bold")
ax.set_xlabel("Dimensión 1")
ax.set_ylabel("Dimensión 2")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for nombre_grupo in color_por_grupo:
    color_grupo = color_por_grupo[nombre_grupo]
    ax.scatter([], [], color=color_grupo, s=70, label=nombre_grupo)

ax.legend(title="Campo", frameon=False, loc="best")
plt.show()


### Cómo leer el gráfico

No busques una separación perfecta. El corpus es chico para eso. Lo que sí conviene mirar es si aparecen cercanías razonables dentro de cada campo: familia, fútbol y escritura.


No conviene leer este gráfico como si fuera una verdad semántica total. Sí conviene leerlo como una transformación importante: **la proximidad entre palabras ya se volvió una relación geométrica**.


---

## 7. Ejemplo práctico: un microcorpus rioplatense

Hasta acá trabajamos con un corpus deliberadamente controlado. Ahora vamos a hacer un desplazamiento breve hacia un microcorpus más situado. No buscamos estabilidad estadística alta: buscamos ver cómo estas ideas también sirven para pensar registros locales y variantes del español rioplatense.


In [ ]:
# --- Microcorpus rioplatense para una prueba breve ---

microcorpus_rioplatense = [
    "En la redacción el laburo salió con la revista digital.",
    "El laburante corrigió la crónica para la revista.",
    "La hinchada azulgrana llenó la tribuna en el Nuevo Gasómetro.",
    "El pibe escribió la crónica del partido azulgrana.",
    "Si tenés tiempo, revisá el relato antes del cierre.",
    "La historia del barrio salió en la revista del club.",
]

microcorpus_rioplatense_tokenizado = tokenizar_corpus(microcorpus_rioplatense, STOPWORDS_MINIMAS)
tabla_coocurrencias_rioplatense = construir_tabla_coocurrencias(microcorpus_rioplatense_tokenizado, tamaño_ventana=2)
tabla_ppmi_rioplatense = calcular_tabla_ppmi(tabla_coocurrencias_rioplatense)

modelo_reduccion_rioplatense = TruncatedSVD(n_components=2, random_state=42)
matriz_densa_rioplatense = modelo_reduccion_rioplatense.fit_transform(tabla_ppmi_rioplatense)

tabla_vectores_rioplatenses = pd.DataFrame(
    matriz_densa_rioplatense,
    index=tabla_ppmi_rioplatense.index,
    columns=["dimensión_1", "dimensión_2"],
)

palabras_rioplatenses_para_vecinos = ["laburo", "azulgrana", "crónica"]

for indice_palabra in range(len(palabras_rioplatenses_para_vecinos)):
    palabra_actual = palabras_rioplatenses_para_vecinos[indice_palabra]

    if palabra_actual not in tabla_vectores_rioplatenses.index:
        print(f"La palabra '{palabra_actual}' no quedó en el vocabulario tokenizado.")
        continue

    tabla_vecinos_actual = buscar_vecinos_semanticos(tabla_vectores_rioplatenses, palabra_actual, top_n=4)
    print(f"Vecinos en el microcorpus rioplatense para '{palabra_actual}'")
    display(tabla_vecinos_actual)


Este ejemplo corto no alcanza para producir una geometría robusta, pero sí deja ver algo importante: cuando pasamos a registros locales, empiezan a aparecer problemas de escala, cobertura y variación morfológica que van a ser centrales en `FastText`.


---

## 8. Hacia Word2Vec, GloVe y FastText

Hasta acá seguimos una ruta basada en conteos:

1. observar contextos;
2. contar coocurrencias;
3. ponderarlas con `PPMI`;
4. comprimir la matriz a vectores densos.

`Word2Vec`, `GloVe` y `FastText` retoman esta misma intuición distribucional, pero no la implementan exactamente del mismo modo.

- **Word2Vec** aprende vectores prediciendo contexto local.
- **GloVe** trabaja de manera más directa con estadísticas globales de coocurrencia.
- **FastText** incorpora subpalabras, lo que mejora el tratamiento de formas raras o morfológicamente complejas.

La ventana de contexto que usamos recién sigue siendo central. Lo que cambia es cómo se aprovecha.


In [ ]:
# --- Pares centro-contexto: la misma ventana, leída como datos de entrenamiento ---

def generar_pares_centro_contexto(tokens_texto, tamaño_ventana=2):
    """Genera pares (centro, contexto) a partir de una lista de tokens y una ventana local."""

    pares_generados = []
    cantidad_tokens = len(tokens_texto)

    for posicion_token in range(cantidad_tokens):
        palabra_central = tokens_texto[posicion_token]
        inicio_ventana = posicion_token - tamaño_ventana
        fin_ventana = posicion_token + tamaño_ventana

        for posicion_contexto in range(inicio_ventana, fin_ventana + 1):
            if posicion_contexto < 0:
                continue

            if posicion_contexto >= cantidad_tokens:
                continue

            if posicion_contexto == posicion_token:
                continue

            palabra_contexto = tokens_texto[posicion_contexto]
            par_actual = {
                "palabra_central": palabra_central,
                "palabra_contexto": palabra_contexto,
            }
            pares_generados.append(par_actual)

    tabla_pares = pd.DataFrame(pares_generados)
    return tabla_pares

texto_ejemplo_word2vec = corpus_controlado_tokenizado[4]
tabla_pares_centro_contexto = generar_pares_centro_contexto(texto_ejemplo_word2vec, tamaño_ventana=2)

print("Texto tokenizado usado para generar pares")
print(texto_ejemplo_word2vec)
print("")
print("Primeros pares centro-contexto")
display(tabla_pares_centro_contexto.head(12))


Ese bloque muestra la continuidad conceptual más importante del cuaderno:

- en un enfoque **count-based**, la ventana llena una matriz de coocurrencias;
- en un enfoque **predictivo**, la ventana genera ejemplos para entrenar un modelo.

Por eso este cuaderno no reemplaza a `Word2Vec`, `FastText` o `GloVe`. Los prepara.


---

## 9. Reflexión guiada, límites y cierre

### Lo que ganamos

En este cuaderno recorriste un pasaje importante:

1. dejaste de mirar palabras como unidades aisladas;
2. empezaste a mirarlas como perfiles de contexto;
3. convertiste esos perfiles en una matriz de coocurrencias;
4. ponderaste la asociación con `PPMI`;
5. comprimiste la representación a un espacio denso;
6. observaste que cierta cercanía semántica puede emerger como cercanía geométrica.

### Límites del experimento

Este cuaderno muestra un mecanismo, no una solución final. Conviene dejar claros varios límites:

- el corpus principal es pequeño y controlado;
- la similitud observada depende del recorte y del tamaño de ventana;
- `PPMI + SVD` no resuelve por sí solo polisemia ni sesgos;
- en corpus chicos, la geometría puede ser inestable;
- que dos palabras queden cerca no autoriza una interpretación automática sin volver al corpus.

### Actividad breve

1. Cambiá el tamaño de ventana de `2` a `1` y después a `3`.
2. Compará cómo cambian los vecinos de `gol` y `relato`.
3. Anotá qué gana y qué pierde el modelo cuando amplía el contexto.

### Preguntas de transición

- ¿Qué comparte `Word2Vec` con la matriz de coocurrencias y qué cambia?
- ¿Por qué `GloVe` puede leerse como una continuación natural de este cuaderno?
- ¿Por qué `FastText` podría comportarse mejor con variantes morfológicas del español?

### Síntesis final

`Bag of Words` y `TF-IDF` representan documentos por términos observados. Las coocurrencias representan palabras por contextos observados. Si esos perfiles de contexto se ponderan y se comprimen, aparecen vectores densos.

Ese es el punto de partida conceptual para entrar con más precisión a `Word2Vec`, `FastText` y `GloVe`.
